## 1. Import Libraries & Load Dataset

**Tujuan:** Memuat dataset hasil cleaning (`stroke_clean.csv`) dari NB03, dan membuang kolom `id` karena bukan fitur prediktif (hanya nomor identifikasi pasien).

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu, pointbiserialr
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

df = pd.read_csv('../data/processed/stroke_clean.csv')
df = df.drop(columns=['id'])

print(f"Shape: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Kolom: {df.columns.tolist()}")

Shape: 5094 baris, 11 kolom
Kolom: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']


**Insight:** Dataset berhasil dimuat dengan shape (5.094 baris, 11 kolom) setelah `id` dibuang, tersisa 10 fitur kandidat + 1 target (`stroke`).

## 2. Persiapan Encoding

**Tujuan:** Membuat versi dataset dengan kolom kategorikal ter-encode menjadi numerik, khusus untuk keperluan Random Forest & Mutual Information di Section 4 (kedua metode ini butuh input numerik). Uji statistik di Section 3 (Chi-square, Mann-Whitney) tetap memakai data kategorikal asli, karena metode tersebut memang bekerja langsung dengan kategori.

In [3]:
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

df_encoded = df.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

gender: {'Female': np.int64(0), 'Male': np.int64(1)}
ever_married: {'No': np.int64(0), 'Yes': np.int64(1)}
work_type: {'Govt_job': np.int64(0), 'Never_worked': np.int64(1), 'Private': np.int64(2), 'Self-employed': np.int64(3), 'children': np.int64(4)}
Residence_type: {'Rural': np.int64(0), 'Urban': np.int64(1)}
smoking_status: {'Unknown': np.int64(0), 'formerly smoked': np.int64(1), 'never smoked': np.int64(2), 'smokes': np.int64(3)}


**Insight:** Lima kolom kategorikal berhasil di-encode menjadi numerik untuk keperluan Random Forest & Mutual Information. Encoding ini bersifat sementara/khusus untuk analisis feature selection — pendekatan encoding final untuk masing-masing algoritma modeling (Random Forest vs Logistic Regression) akan ditentukan ulang secara terpisah di NB08.

## 3. Filter Method — Uji Statistik

**Tujuan:** Menguji hubungan setiap fitur terhadap target `stroke` secara univariate (satu-satu), menggunakan uji yang sesuai dengan tipe datanya masing-masing:
- **Chi-square**: untuk fitur kategorikal/biner (gender, hypertension, heart_disease, ever_married, work_type, Residence_type, smoking_status) vs target kategorikal (stroke)
- **Mann-Whitney U**: untuk fitur numerik (age, avg_glucose_level, bmi) vs target — dipilih non-parametrik (bukan t-test) karena distribusi `avg_glucose_level` diketahui skewed/bimodal
- **Point-biserial correlation**: melengkapi Mann-Whitney dengan mengukur arah & kekuatan hubungan fitur numerik terhadap target biner

In [4]:
target = 'stroke'
cat_cols = ['gender', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
num_cols = ['age', 'avg_glucose_level', 'bmi']

print("=== CHI-SQUARE: fitur kategorikal vs stroke ===")
print(f"{'Fitur':<18}{'Chi2':<12}{'p-value':<15}{'Signifikan?'}")
for col in cat_cols:
    ct = pd.crosstab(df[col], df[target])
    chi2, p, dof, exp = chi2_contingency(ct)
    sig = "YA" if p < 0.05 else "TIDAK"
    print(f"{col:<18}{chi2:<12.2f}{p:<15.6f}{sig}")

print("\n=== MANN-WHITNEY U & POINT-BISERIAL: fitur numerik vs stroke ===")
print(f"{'Fitur':<20}{'U-stat':<15}{'p-value':<15}{'r (korelasi)':<15}{'Signifikan?'}")
for col in num_cols:
    g0 = df[df[target]==0][col]
    g1 = df[df[target]==1][col]
    u, p_mw = mannwhitneyu(g0, g1)
    corr, p_corr = pointbiserialr(df[target], df[col])
    sig = "YA" if p_mw < 0.05 else "TIDAK"
    print(f"{col:<20}{u:<15.1f}{p_mw:<15.6f}{corr:<15.4f}{sig}")

=== CHI-SQUARE: fitur kategorikal vs stroke ===
Fitur             Chi2        p-value        Signifikan?
gender            0.57        0.449409       TIDAK
hypertension      79.90       0.000000       YA
heart_disease     92.30       0.000000       YA
ever_married      57.40       0.000000       YA
work_type         47.29       0.000000       YA
Residence_type    1.47        0.225128       TIDAK
smoking_status    26.34       0.000008       YA

=== MANN-WHITNEY U & POINT-BISERIAL: fitur numerik vs stroke ===
Fitur               U-stat         p-value        r (korelasi)   Signifikan?
age                 198073.5       0.000000       0.2438         YA
avg_glucose_level   464652.0       0.000000       0.1316         YA
bmi                 514135.5       0.000261       0.0365         YA


**Insight:** Uji statistik mengonfirmasi `gender` (p=0.449) dan `Residence_type` (p=0.225) sebagai satu-satunya fitur yang gagal signifikan terhadap `stroke` — kandidat kuat untuk dibuang. Seluruh fitur lain signifikan (p<0.05), dengan `age` menunjukkan korelasi terkuat (r=0.244), diikuti `avg_glucose_level` (r=0.132). Menariknya, `bmi` signifikan secara statistik namun korelasi linearnya sangat lemah (r=0.037) — ini mengindikasikan kemungkinan hubungan nonlinear yang perlu dikonfirmasi lewat Random Forest importance di Section 4, karena point-biserial correlation hanya mampu mendeteksi hubungan linear.

## 4. Embedded Method — Random Forest Importance & Mutual Information

**Tujuan:** Melengkapi hasil uji statistik Section 3 (yang cuma melihat 1 fitur vs target secara terpisah) dengan metode yang memperhitungkan fitur dalam konteks bersama (Random Forest) dan yang bisa menangkap hubungan non-linear (Mutual Information) — khususnya untuk mengonfirmasi dugaan soal `bmi` yang lemah secara korelasi linear.

In [5]:
X = df_encoded.drop(columns=['stroke'])
y = df_encoded['stroke']

rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("=== RANDOM FOREST FEATURE IMPORTANCE ===")
print(importances)

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print("\n=== MUTUAL INFORMATION SCORE ===")
print(mi_series)

=== RANDOM FOREST FEATURE IMPORTANCE ===
age                  0.386375
avg_glucose_level    0.204777
bmi                  0.182193
smoking_status       0.051394
work_type            0.047539
ever_married         0.033626
hypertension         0.029818
gender               0.022758
Residence_type       0.021630
heart_disease        0.019889
dtype: float64

=== MUTUAL INFORMATION SCORE ===
age                  0.036067
hypertension         0.012080
bmi                  0.010210
ever_married         0.010130
heart_disease        0.009861
avg_glucose_level    0.005924
work_type            0.003792
smoking_status       0.003780
Residence_type       0.002894
gender               0.000000
dtype: float64


## 5. Sintesis & Keputusan Final

**Tujuan:** Menggabungkan seluruh bukti dari filter method (Section 3) dan embedded method (Section 4), dikombinasikan dengan domain knowledge, untuk menetapkan daftar fitur final. Fitur final kemudian disimpan sebagai dataset baru untuk digunakan pada tahap-tahap berikutnya (NB06-NB08).

**Ringkasan keputusan:**
- **Dipertahankan** (signifikan secara statistik + relevan secara medis): `age`, `avg_glucose_level`, `bmi`, `hypertension`, `heart_disease`, `smoking_status`
- **Dibuang** (tidak signifikan di seluruh uji, termasuk Mutual Information = 0 untuk gender): `gender`, `Residence_type`
- **Dibuang** (signifikan secara statistik, namun diduga kuat sebagai proxy dari `age` berdasarkan RF importance moderat dan tidak adanya mekanisme klinis langsung ke stroke): `ever_married`, `work_type`

In [6]:
final_features = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'smoking_status']

df_selected = df[final_features + ['stroke']].copy()

print(f"Shape sebelum seleksi: {df.shape[1]} kolom")
print(f"Shape sesudah seleksi: {df_selected.shape[1]} kolom")
print(f"\nFitur final: {final_features}")

df_selected.to_csv('../data/processed/stroke_selected_features.csv', index=False)
print(f"\nFile berhasil disimpan: data/processed/stroke_selected_features.csv")

check = pd.read_csv('../data/processed/stroke_selected_features.csv')
print(f"Verifikasi shape hasil baca ulang: {check.shape[0]} baris, {check.shape[1]} kolom")

Shape sebelum seleksi: 11 kolom
Shape sesudah seleksi: 7 kolom

Fitur final: ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'smoking_status']

File berhasil disimpan: data/processed/stroke_selected_features.csv
Verifikasi shape hasil baca ulang: 5094 baris, 7 kolom


## 6. Kesimpulan

Berdasarkan kombinasi uji statistik (Chi-square, Mann-Whitney U, point-biserial correlation) dan model-based importance (Random Forest, Mutual Information), ditetapkan 6 fitur final untuk tahap modeling: `age`, `avg_glucose_level`, `bmi`, `hypertension`, `heart_disease`, dan `smoking_status`.

Dua fitur (`gender`, `Residence_type`) dibuang karena gagal signifikan di seluruh uji statistik, termasuk Mutual Information sebesar 0 untuk `gender` — mengonfirmasi tidak adanya informasi yang berguna dari fitur ini terhadap prediksi stroke.

Dua fitur lain (`ever_married`, `work_type`) meskipun signifikan secara statistik, diputuskan untuk tidak disertakan karena nilainya sangat berkorelasi dengan `age` (contoh: seluruh individu berstatus "belum menikah" berusia di bawah 18 tahun), sehingga informasinya sebagian besar tumpang tindih dengan fitur `age` yang sudah dipertahankan. Keputusan ini mengutamakan kesederhanaan model dan menghindari risiko multikolinearitas pada tahap Logistic Regression (NB10).

Wrapper method (RFE) sengaja tidak digunakan karena filter method dan embedded method telah saling mengonfirmasi hasil yang konsisten, sesuai arahan tugas untuk memilih metode berdasarkan kebutuhan dataset dengan argumen yang jelas.

Dataset final (6 fitur + target) disimpan sebagai `stroke_selected_features.csv` untuk digunakan pada tahap dimensionality reduction (NB06), visualisasi (NB07), dan persiapan data modeling (NB08).